# Pipeline de Inferencia Batch para Predicción de Suscripción y ROI

## Introducción

Este notebook tiene como propósito cargar un modelo de predicción de suscripción previamente entrenado, obtener un batch de características de clientes recientes, generar predicciones de suscripción para estos clientes, calcular el Retorno de la Inversión (ROI) estimado para una campaña basada en estas predicciones y, finalmente, guardar los resultados.

## 1. Importar Bibliotecas

In [ ]:
import pandas as pd
import numpy as np
import os
import datetime

import tensorflow as tf
import mlflow
import mlflow.keras # o mlflow.tensorflow si se usó ese para loguear

print(f"TensorFlow Version: {tf.__version__}")
print(f"MLflow Version: {mlflow.__version__}")

## 2. Definición de Parámetros de ROI (Reiteración)

Para calcular el ROI de la campaña, utilizamos los siguientes parámetros asumidos:

*   `R_sub`: Ingreso promedio esperado por cada suscripción exitosa (ej. $100 MXN).
*   `C_contact`: Costo promedio por cada contacto realizado a un cliente (ej. $1 MXN).

La fórmula para el ROI de la campaña es:

```
ROI_campaña = (Ingreso_Total_Predicho - Costo_Total_Campaña) / Costo_Total_Campaña
```

Donde:
*   `Ingreso_Total_Predicho = SUMA(probabilidad_suscripcion_cliente * R_sub)` para todos los clientes en el batch.
*   `Costo_Total_Campaña = SUMA(contactos_realizados_al_cliente * C_contact)` para todos los clientes en el batch. La columna `campaign` del dataset original nos da el número de contactos.

In [ ]:
# Parámetros para el cálculo de ROI
R_sub = 100  # Ingreso por suscripción
C_contact = 1 # Costo por contacto

## 3. Cargar Modelo Entrenado desde MLflow

Cargamos el modelo Keras que fue entrenado y registrado usando MLflow en el pipeline de entrenamiento.

In [ ]:
mlflow_experiment_name = "Bank_Subscription_Prediction_DNN_v1" # Mismo nombre que en training_pipeline
model = None

try:
    # Obtener el experimento por nombre
    experiment = mlflow.get_experiment_by_name(mlflow_experiment_name)
    if experiment is None:
        raise Exception(f"Experimento '{mlflow_experiment_name}' no encontrado.")
    
    # Listar todas las runs del experimento, ordenadas por fecha de inicio (la más reciente primero)
    runs_df = mlflow.search_runs(experiment_ids=experiment.experiment_id, order_by=["start_time DESC"])
    
    if runs_df.empty:
        raise Exception(f"No runs encontradas en el experimento '{mlflow_experiment_name}'.")
    
    # Obtener el ID de la run más reciente (primera fila después de ordenar)
    latest_run_id = runs_df.iloc[0]['run_id']
    print(f"Run ID más reciente seleccionada: {latest_run_id}")
    
    # Construir el URI del modelo
    # El nombre 'model' es el artifact_path por defecto si se usó autolog o log_model sin un path específico.
    # Si se usó un nombre de carpeta específico (ej. 'keras-dnn-model'), usar ese.
    # mlflow.tensorflow.autolog() suele guardar en una carpeta llamada 'model'.
    model_uri = f"runs:/{latest_run_id}/model"
    print(f"Cargando modelo desde URI: {model_uri}")
    
    model = mlflow.keras.load_model(model_uri)
    print("\nModelo cargado exitosamente desde MLflow.")
    model.summary() # Mostrar resumen del modelo para confirmar
    
except Exception as e:
    print(f"Error al cargar el modelo desde MLflow: {e}")
    print("Asegúrate de que el servidor de MLflow esté accesible (si es remoto) y que el nombre del experimento y la run sean correctos.")

## 4. Cargar Batch de Características Recientes

Cargamos el archivo `processed_bank_data.parquet` generado por `feature_pipeline.ipynb`. Para simular un batch de inferencia, tomaremos una muestra de estos datos (e.g., las primeras 1000 filas).

**Suposición importante:** Se asume que el `processed_bank_data.parquet` contiene:
1.  Todas las características procesadas (escaladas y codificadas one-hot) listas para el modelo.
2.  La columna `y` original (aunque no se usa para la predicción, puede ser útil para análisis comparativos).
3.  Una columna con los valores originales de `campaign` (número de contactos), que llamaremos `campaign_original`, necesaria para el cálculo del costo de la campaña.

In [ ]:
processed_data_path = '../feature_pipeline/data/processed_bank_data.parquet'
df_full_processed = None
X_batch_processed = None
y_batch_true = None # Objetivo real, para referencia si está disponible
campaign_values_batch = None # Columna 'campaign' original para el batch
batch_indices = None # Para mantener los índices originales del batch

if not os.path.exists(processed_data_path):
    print(f"Error: El archivo de datos procesados '{processed_data_path}' no se encontró.")
    print("Asegúrate de que el notebook 'feature_pipeline.ipynb' se haya ejecutado correctamente y la ruta sea la correcta.")
else:
    print(f"Cargando datos procesados desde: {processed_data_path}")
    df_full_processed = pd.read_parquet(processed_data_path)
    print("Datos procesados cargados exitosamente.")
    
    # Verificar columnas necesarias
    required_cols_for_roi = ['campaign_original', 'y'] # Asumiendo que 'y' y 'campaign_original' están en el parquet.
    missing_cols = [col for col in required_cols_for_roi if col not in df_full_processed.columns]
    if missing_cols:
        print(f"Error: Faltan las siguientes columnas necesarias en 'processed_bank_data.parquet': {missing_cols}")
        print("El pipeline de características debe asegurar que 'campaign_original' (valores crudos de 'campaign') se guarde.")
        df_full_processed = None # Invalidar para prevenir errores posteriores
    else:
        # Simulación de un batch reciente: tomar las primeras 1000 filas
        batch_size = 1000
        df_batch = df_full_processed.head(batch_size).copy() # Usar .copy() para evitar SettingWithCopyWarning
        batch_indices = df_batch.index
        print(f"\nSimulando un batch reciente con las primeras {batch_size} filas.")
        
        # Separar características procesadas para el modelo, el objetivo real y los valores de campaña
        # Las características para el modelo son todas las columnas excepto 'y' y 'campaign_original'
        y_batch_true = df_batch['y']
        campaign_values_batch = df_batch['campaign_original']
        X_batch_processed = df_batch.drop(columns=['y', 'campaign_original'])
        
        print(f"Forma de X_batch_processed (características para el modelo): {X_batch_processed.shape}")
        print(f"Forma de y_batch_true (objetivo real del batch): {y_batch_true.shape}")
        print(f"Forma de campaign_values_batch (contactos originales del batch): {campaign_values_batch.shape}")
        display(X_batch_processed.head())

## 5. Generación de Predicciones

Utilizamos el modelo cargado para generar predicciones de probabilidad de suscripción para el batch de clientes.

In [ ]:
pred_probabilidades = None

if model is not None and X_batch_processed is not None:
    print("Generando predicciones de probabilidad...")
    pred_probabilidades = model.predict(X_batch_processed)
    # La salida de Keras es (n_samples, 1), la aplanamos a (n_samples,)
    pred_probabilidades = pred_probabilidades.flatten()
    print(f"Predicciones generadas. Forma: {pred_probabilidades.shape}")
    print("Primeras 5 probabilidades predichas:", pred_probabilidades[:5])
    
    # Añadir las probabilidades al DataFrame del batch (o a uno nuevo para resultados)
    # Crear un DataFrame para los resultados del batch
    df_batch_results = pd.DataFrame({
        'id_cliente': batch_indices, # Asumiendo que el índice es el identificador
        'probabilidad_suscripcion': pred_probabilidades,
        'contactos_campaña_original': campaign_values_batch,
        'suscripcion_real': y_batch_true # Para referencia
    })
    display(df_batch_results.head())
else:
    print("No se pueden generar predicciones. El modelo o el batch de características no están disponibles.")
    df_batch_results = None

## 6. Cálculo del ROI de la Campaña

Con las probabilidades de suscripción predichas y los parámetros de ROI definidos, calculamos el ROI estimado para esta campaña simulada sobre el batch.

In [ ]:
if df_batch_results is not None:
    # Ingreso total predicho para el batch
    # Cada cliente contribuye con (probabilidad_suscripcion * R_sub)
    ingreso_total_predicho = (df_batch_results['probabilidad_suscripcion'] * R_sub).sum()
    
    # Costo total de la campaña para el batch
    # Cada cliente contribuye con (contactos_realizados * C_contact)
    costo_total_campaña = (df_batch_results['contactos_campaña_original'] * C_contact).sum()
    
    print(f"Ingreso Total Predicho para el Batch: ${ingreso_total_predicho:,.2f} MXN")
    print(f"Costo Total de la Campaña para el Batch: ${costo_total_campaña:,.2f} MXN")
    
    # Calcular ROI
    if costo_total_campaña > 0:
        roi_campaña = (ingreso_total_predicho - costo_total_campaña) / costo_total_campaña
        print(f"ROI Estimado de la Campaña para el Batch: {roi_campaña:.2%}")
    else:
        roi_campaña = 0
        print("El costo total de la campaña es cero, no se puede calcular el ROI (se asume 0).")
else:
    print("No se pueden calcular las métricas de ROI porque df_batch_results no está definido.")

## 7. Almacenamiento de Predicciones y ROI

Guardamos las predicciones del batch en un archivo CSV. El ROI general de la campaña (calculado arriba) se podría registrar en un sistema de monitoreo o base de datos junto con metadatos de la ejecución.

In [ ]:
if df_batch_results is not None:
    # Crear un directorio para guardar los resultados si no existe
    output_dir = './batch_inference_outputs/'
    if not os.path.exists(output_dir):
        os.makedirs(output_dir)
        print(f"Directorio de salida '{output_dir}' creado.")
        
    # Generar un nombre de archivo con timestamp
    timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
    predictions_filename = f"batch_predictions_{timestamp}.csv"
    full_predictions_path = os.path.join(output_dir, predictions_filename)
    
    try:
        df_batch_results.to_csv(full_predictions_path, index=False)
        print(f"Predicciones del batch guardadas en: '{full_predictions_path}'")
    except Exception as e:
        print(f"Error al guardar las predicciones: {e}")
else:
    print("No hay resultados de predicciones para guardar.")

# El ROI ya se imprimió. En un sistema real, se podría loguear a MLflow como una métrica de post-entrenamiento 
# o a un dashboard específico de seguimiento de campañas.

Las predicciones y el ROI calculado se guardarían en una base de datos o en un sistema de almacenamiento (como S3 o Google Cloud Storage) para que los servicios downstream puedan consumirlos (por ejemplo, para alimentar dashboards de BI, sistemas de CRM, o para análisis posteriores).

## 8. Automatización (Comentario)

Este notebook está diseñado para ser ejecutado en un horario regular (e.g., cada hora, diariamente, etc.) mediante una herramienta de orquestación como Apache Airflow, Kubeflow Pipelines, o una GitHub Action para generar predicciones sobre nuevos datos de clientes que lleguen en batch. La frecuencia dependerá de la necesidad del negocio y la disponibilidad de nuevos datos.

## 9. Generación de `requirements.txt`

Ejecuta la siguiente celda para imprimir las versiones de las bibliotecas clave utilizadas en este notebook. Copia esta salida a un archivo `requirements.txt` para asegurar la reproducibilidad del entorno.

In [ ]:
print("# --- requirements.txt para batch_inference_pipeline.ipynb ---")
print(f"pandas=={pd.__version__}")
print(f"numpy=={np.__version__}")
print(f"tensorflow=={tf.__version__}")
print(f"mlflow=={mlflow.__version__}")
print(f"pyarrow") # Necesario para pd.read_parquet